# 03_tinyvec: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/03_tinyvec/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Walk through TinyVec storage and mutation.

Concepts: stable IDs, replacement, deletion, exact scoring.
Run: uv run python modules/03_tinyvec/examples/01_from_scratch.py
Expected: same ID updated once; deleted ID absent
CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

import time

import numpy as np

from common.pretty import show_hits
from common.stores.registry import make_store
from common.stores.types import CollectionSpec, Record


def main() -> None:
    store = make_store("tinyvec")
    store.create_collection(CollectionSpec("desk", 2, "cosine", "hand-v1"))
    try:
        store.insert(
            [
                Record(
                    "warmer",
                    np.array([1, 0.1]),
                    {"title": "Ceramic mug warmer", "price": 30, "region": "north"},
                )
            ]
        )
        # KEY IDEA: upsert replaces the whole record for the same ID; it does not append another copy.
        store.upsert(
            [
                Record(
                    "warmer",
                    np.array([1, 0.1]),
                    {"title": "Ceramic mug warmer", "price": 25, "region": "north"},
                )
            ]
        )
        start = time.perf_counter()
        hits = store.search([1, 0], space="hand-v1")
        elapsed = (time.perf_counter() - start) * 1000
        assert len(hits) == 1 and hits[0].metadata["price"] == 25
        show_hits(hits, elapsed, title="Updated product")
        store.delete(["warmer"])
        assert store.search([1, 0]) == []
        print("Deleted ID absent; empty search is valid.")
    finally:
        store.close()


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## 02_production

Source: `modules/03_tinyvec/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Use Chroma IDs, metadata, persistence, and CRUD.

    Concepts: persistent collection, explicit embeddings, replace/delete/filter.
    Run: uv run --extra embedded python modules/03_tinyvec/examples/02_production.py
    Expected: one north-region item after update and delete; reopen count 1
    CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
    Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
    """

    # course-extra: embedded
    import uuid

    import chromadb

    from common.settings import cache_dir

    def main() -> None:
        path = cache_dir() / "tier1" / ("chroma-" + uuid.uuid4().hex)
        client = chromadb.PersistentClient(
            path=str(path), settings=chromadb.Settings(anonymized_telemetry=False)
        )
        collection = client.create_collection(
            "desk", embedding_function=None, configuration={"hnsw": {"space": "cosine"}}
        )
        collection.upsert(
            ids=["warmer", "flask"],
            embeddings=[[1.0, 0.1], [0.8, 1.0]],
            metadatas=[{"region": "north", "price": 30}, {"region": "south", "price": 25}],
        )
        collection.upsert(
            ids=["warmer"], embeddings=[[1.0, 0.1]], metadatas=[{"region": "north", "price": 20}]
        )
        found = collection.query(
            query_embeddings=[[1.0, 0.0]], n_results=2, where={"region": "north"}
        )
        assert found["ids"] == [["warmer"]]
        collection.delete(ids=["flask"])
        # KEY IDEA: metadata and IDs live in the engine; the model still belongs to the application.
        reopened = chromadb.PersistentClient(
            path=str(path), settings=chromadb.Settings(anonymized_telemetry=False)
        )
        assert reopened.get_collection("desk", embedding_function=None).count() == 1
        print("Reopened count: 1; persistent files:", path)
        print(
            "This is same-process reopen; a fresh-process restore check is part of the checkpoint."
        )

    if __name__ == "__main__":
        from common.errors import run_cli

        run_cli(main)

## 03_measure

Source: `modules/03_tinyvec/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Show why filtering only the first few hits can lose valid results.

Concepts: prefilter, postfilter, candidate budget, filter recall.
Run: uv run python modules/03_tinyvec/examples/03_measure.py
Expected: postfilter returns 0; prefilter returns 1
CPU: about 2–10 seconds. Data: included Marlowe vectors or explicit hand features.
Models: no download; optional --live loads cached MiniLM where documented. Docker: none.
"""

import numpy as np

from common.stores.registry import make_store
from common.stores.types import CollectionSpec, Record
from common.teaching import evidence, timed


def main() -> None:
    store = make_store()
    store.create_collection(CollectionSpec("filter-demo", 2, "cosine", "hand-v1"))
    rows = [
        Record("warmer-unavailable", np.array([1.0, 0.0]), {"stock": 0}),
        Record("warmer-available", np.array([0.9, 0.1]), {"stock": 4}),
    ]
    store.upsert(rows)
    expression = {"op": "gt", "field": "stock", "value": 0}
    try:
        post = [h for h in store.search([1, 0], 1) if h.metadata["stock"] > 0]
        pre = store.search([1, 0], 1, expression)
        assert not post and [h.id for h in pre] == ["warmer-available"]
        # KEY IDEA: eligibility is part of the nearest-neighbor question.
        evidence(
            "filter_order",
            {
                "postfilter_count": len(post),
                "prefilter_count": len(pre),
                "prefilter": timed(lambda: store.search([1, 0], 1, expression)),
            },
        )
        print("postfilter returns 0; prefilter returns 1")
    finally:
        store.close()


if __name__ == "__main__":
    from common.errors import run_cli

    run_cli(main)

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.apps.search import open_catalog, search_saved

store, _, _ = open_catalog(limit=100)
try:
    _, hits, _ = search_saved(store, expression={"op": "lte", "field": "price", "value": 30})
    print([(h.id, h.metadata["price"]) for h in hits])
finally:
    store.close()

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.